In [2]:
from fitsio import FITS
import fitsio
import matplotlib.pyplot as plt
import astropy.io.fits as pyfits
import numpy as np
import os
from matplotlib.pyplot import rcParams
import matplotlib._color_data as mcd
import math
rcParams['figure.figsize'] = 10, 5
rcParams['lines.linewidth'] = 2
rcParams['axes.labelsize'] = 15
rcParams['legend.fontsize'] = 12

import h5py
from scipy import interpolate
#from picca import wedgize

# Get mock DLA

In [3]:
from desidlas.datasets.preprocess import estimate_s2n,normalize,rebin
from desidlas.datasets.DesiMock import DesiMock
from desidlas.dla_cnn.defs import best_v
import numpy as np
import os
from os.path import join
from pkg_resources import resource_filename
from pathlib import Path
from desidlas.datasets.get_sightlines import get_sightlines

In [9]:
### Set directory to desi spectra:
spectrapath='/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/spectra-16'
### Set saving path for sightlines:
savepath='/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/sightlines'
### Run DLA finder to get sightlines:
item1 = os.listdir(spectrapath)
for k in item1:
    itemlist=os.listdir(spectrapath+'/'+str(k))
    savedir=savepath+'/'+str(k)
    if not os.path.isdir(savedir):
        os.mkdir(savedir)
    #item2=['10000']
    for j in itemlist:
        try: 
            datafile_path=spectrapath+'/'+str(k)+'/'+str(j)
            spectra= os.path.join(datafile_path, 'spectra-16-{}.fits'.format(j))
            truth=[]
            zbest=os.path.join(datafile_path,  'zbest-16-{}.fits'.format(j))

            outpath=savedir+'/'+'sightlines-%s.npy'%j
            #print(spectra,truth,zbest,outpath)
            sightlines=get_sightlines(spectra,truth,zbest,outpath)
        except:
            print('fail ',spectra,truth,zbest,outpath)
            continue
    print('%s done'%k)


"\nfor k in item1:\n    itemlist=os.listdir(spectrapath+'/'+str(k))\n    savedir=savepath+'/'+str(k)\n    if not os.path.isdir(savedir):\n        os.mkdir(savedir)\n    #item2=['10000']\n    for j in itemlist:\n        try: \n            datafile_path=spectrapath+'/'+str(k)+'/'+str(j)\n            spectra= os.path.join(datafile_path, 'spectra-16-{}.fits'.format(j))\n            truth=[]\n            zbest=os.path.join(datafile_path,  'zbest-16-{}.fits'.format(j))\n\n            outpath=savedir+'/'+'sightlines-%s.npy'%j\n            print(spectra,truth,zbest,outpath)\n            sightlines=get_sightlines(spectra,truth,zbest,outpath)\n        except:\n            continue\n    print('%s done'%k)\n"

# Arrange all the sightlines 

In [3]:
sightline_all = []
### Set directory to desi spectra:
spectrapath='/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/spectra-16'
### Set saving path for sightlines:
savepath='/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/sightlines'
item1 = os.listdir(savepath)
for k in item1:
    itemlist=os.listdir(savepath+'/'+str(k))
    for j in itemlist:
        sightline_all=sightline_all+np.load(savepath+'/'+str(k)+'/'+str(j),allow_pickle = True,encoding='latin1').tolist()

In [4]:
len(sightline_all)

31390

In [4]:
import scipy.signal as signal
from desidlas.datasets.datasetting import split_sightline_into_samples,select_samples_50p_pos_neg,pad_sightline
from desidlas.datasets.preprocess import label_sightline
from desidlas.dla_cnn.spectra_utils import get_lam_data
from desidlas.datasets.get_dataset import make_datasets,make_smoothdatasets

In [ ]:
dataset=make_datasets(sightline_all,validate=True,output='/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/dlafinder/dataset.npy')

/global/u1/t/tanting/DESI_analysis/desi-dlas/desidlas/datasets/datasetting.py:85: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  fluxes_matrix = np.vstack(map(lambda x:x[0][x[1]-kernelrangepx:x[1]+kernelrangepx],zip(itertools.repeat(flux_padded), np.nonzero(ix_dla_range)[0]+pixel_num_left)))
/global/u1/t/tanting/DESI_analysis/desi-dlas/desidlas/datasets/datasetting.py:86: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  lam_matrix = np.vstack(map(lambda x:x[0][x[1]-kernelrangepx:x[1]+kernelrangepx],zip(itertools.repeat(lam_padded), np.nonzero(ix_dla_range)[0]+pixel_num_left)))


In [4]:
from desidlas.prediction.pred_sightline import get_results,save_pred
from desidlas.prediction.dla_catalog import catalog_fits
sightlines=np.load('/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/sightlines/5/sightlines-512.npy',allow_pickle = True,encoding='latin1')
dataset=make_datasets(sightlines,validate=True,output='/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/dlafinder/dataset_test.npy')

/global/u1/t/tanting/DESI_analysis/desi-dlas/desidlas/datasets/datasetting.py:85: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  fluxes_matrix = np.vstack(map(lambda x:x[0][x[1]-kernelrangepx:x[1]+kernelrangepx],zip(itertools.repeat(flux_padded), np.nonzero(ix_dla_range)[0]+pixel_num_left)))
/global/u1/t/tanting/DESI_analysis/desi-dlas/desidlas/datasets/datasetting.py:86: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  lam_matrix = np.vstack(map(lambda x:x[0][x[1]-kernelrangepx:x[1]+kernelrangepx],zip(itertools.repeat(lam_padded), np.nonzero(ix_dla_range)[0]+pixel_num_left)))


In [12]:
!python desidlas/prediction/get_partprediction.py -p '/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/dlafinder/dataset_test.npy' -o '/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/dlafinder/partpre_test.npy' -model high

2022-04-07 01:54:34.516659: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /global/u1/t/tanting/DESI_analysis/theoretical_pk/fisher_forecast/openmpi-4.1.2/lib:/opt/cray/pe/papi/6.0.0.12/lib64:/global/common/software/desi/cori/desiconda//20211217-2.0.0/aux/lib:/global/u1/t/tanting/DESI_analysis/theoretical_pk/fisher_forecast/openmpi-4.1.2/lib:/global/u1/t/tanting/DESI_analysis/theoretical_pk/fisher_forecast/openmpi-4.1.2/lib:/global/u1/t/tanting/DESI_analysis/theoretical_pk/fisher_forecast/openmpi-4.1.2/lib:/global/u1/t/tanting/DESI_analysis/theoretical_pk/fisher_forecast/openmpi-4.1.2/lib:/opt/gcc/11.2.0/snos/lib64
2022-04-07 01:54:34.516709: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
[[0.0005, 2e-05, 0.0005, 0.0007, 0.001, 0.003,

In [16]:
### Get DLA catalog and quasar catalog:
from desidlas.prediction.pred_sightline import get_results,save_pred
from desidlas.prediction.dla_catalog import catalog_fits
sightlines=np.load('/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/sightlines/5/sightlines-512.npy',allow_pickle = True,encoding='latin1')
# make real DLA catalog and QSO catalog for mock spectra
real_catalog=catalog_fits(sightlines,dlafile='/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/dlafinder/dla_catalog_test.fits',qsofile='/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/dlafinder/qso_catalog_test.fits')
#real_catalog=catalog_fits(sightlines,dlafile='/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/dlafinder/dla_catalog.fits',qsofile='/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/dlafinder/qso_catalog.fits')

In [17]:
file_dla_test = FITS('/global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/dlafinder/dla_catalog_test.fits')

In [19]:
file_dla_test[1]


  file: /global/cfs/cdirs/desi/users/hiramk/desi/everest/main/mock/london/v9.0.0/everest_main-0.134/dlafinder/dla_catalog_test.fits
  extension: 1
  type: BINARY_TBL
  extname: DLACAT
  rows: 0
  column info:
    TARGET_RA           f8  
    TARGET_DEC          f8  
    ZQSO                f8  
    Z                   f8  
    TARGETID            i8  
    S/N                 f8  
    DLAID               S1  
    NHI                 f8  
    DLA_CONFIDENCE      f8  
    NHI_STD             f8  
    ABSORBER_TYPE       S1  